# LLaMA2-7B W-BFP4 / Top-2 A-BiE4 + DEWA + Conditional FP-ACC PPL

This notebook evaluates the final agreed numerical datapath:

- Decoder Linear weights use single-exponent BFP4 G16/E5.
- Activations use signed-mu+3sigma BiE4 and encode at most two outliers per G16.
- Normal products enter the current-value DEWA accumulator.
- The zero-to-two outlier products are reduced first, then compared with the pre-update DEWA exponent.
- A small outlier partial is skipped; every other nonzero outlier partial enters an FP32 FP-ACC.
- Final Linear output is FP16(DEWA + FP-ACC + bias).
- Only the final conditional-skip PPL path is swept for T = 8, 9, 10, 11, 12.


In [ ]:
%pip install -q "transformers==5.13.1" "datasets==4.0.0" accelerate sentencepiece tqdm


In [ ]:
import gc
import json
import math
import os
import platform
import time
import zipfile
from dataclasses import asdict, dataclass
from getpass import getpass
from pathlib import Path

import datasets
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import triton
import triton.language as tl
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'meta-llama/Llama-2-7b-hf'
DATASET_ID = 'Salesforce/wikitext'
DATASET_CONFIG = 'wikitext-2-raw-v1'
SPLIT = 'test'
CONTEXT_LENGTH = 2048
STRIDE = 2048
DROP_REMAINDER = True
EVALUATION_PROTOCOL = 'non_overlapping_2048_drop_remainder'

FP16_BASELINE_PPL = 5.472103118896484
TOP2_BASELINE_PPL = 6.0155558586120605
TOP2_BASELINE_RESULT = (
    'results/ppl/activation-bie-top2/llama2-7b/'
    'w-bfp4-a-bie4-top2cap-g16-signed-mu3sigma-no-lm-head-s2048.json'
)
EXPECTED_LINEAR_LAYERS = 224
EXPECTED_EVALUATED_BLOCKS = 166
EXPECTED_LOSS_TOKENS = 339_802
EXPECTED_WEIGHT_VALUES = 6_476_005_376
EXPECTED_ACTIVATION_VALUES = 387_117_481_984
T_SWEEP = tuple(range(8, 13))


@dataclass(frozen=True)
class HybridConfig:
    block_size: int = 16
    shared_exponent_bits: int = 5
    mantissa_bits: int = 3
    rounding: str = 'nearest'
    activation_threshold_method: str = 'signed_mean_plus_sigma_k_std'
    sigma_k: float = 3.0
    max_outliers_per_block: int = 2
    topk_tie_break: str = 'lowest_k_index'
    weight_chunk_rows: int = 128
    activation_chunk_rows: int = 2048
    quantize_lm_head: bool = False

    def validate(self):
        if self.block_size != 16:
            raise ValueError('This notebook is fixed to Group-16.')
        if self.shared_exponent_bits != 5:
            raise ValueError('This notebook is fixed to signed E5 shared exponents.')
        if self.mantissa_bits != 3:
            raise ValueError('BFP4/BiE4 require 1 sign bit + 3 magnitude bits.')
        if self.rounding != 'nearest':
            raise ValueError('This notebook is fixed to nearest rounding.')
        if self.activation_threshold_method != 'signed_mean_plus_sigma_k_std':
            raise ValueError('Unexpected threshold method.')
        if not math.isfinite(self.sigma_k) or self.sigma_k < 0:
            raise ValueError('sigma_k must be finite and non-negative.')
        if self.max_outliers_per_block != 2:
            raise ValueError('This notebook is fixed to at most two outliers per block.')
        if self.topk_tie_break != 'lowest_k_index':
            raise ValueError('Unexpected Top-2 tie-break policy.')
        if min(self.weight_chunk_rows, self.activation_chunk_rows) <= 0:
            raise ValueError('Chunk sizes must be positive.')


@dataclass(frozen=True)
class DEWAConfig:
    threshold_bits: int = 8
    exception_threshold_bits: int = 8
    enabled: bool = True
    exception_skip_enabled: bool = True
    exponent_source: str = 'current_numeric_accumulator_leading_exponent'
    exception_reference: str = 'pre_update_dewa_accumulator'

    def validate(self):
        if self.threshold_bits <= 0:
            raise ValueError('threshold_bits must be positive.')
        if self.exception_threshold_bits <= 0:
            raise ValueError('exception_threshold_bits must be positive.')
        if self.exponent_source != 'current_numeric_accumulator_leading_exponent':
            raise ValueError('DEWA must use the current numeric accumulator exponent.')
        if self.exception_reference != 'pre_update_dewa_accumulator':
            raise ValueError('Exception admission must use pre-update DEWA state.')


@dataclass(frozen=True)
class KernelConfig:
    group_k: int = 16
    block_m: int = 32
    block_n: int = 64
    num_warps: int = 4
    num_stages: int = 2

    def validate(self):
        if self.group_k != 16:
            raise ValueError('Kernel group_k must match BiE Group-16.')
        if self.group_k % 8:
            raise ValueError('Packed tags require group_k divisible by 8.')


FORMAT = HybridConfig()
KERNEL = KernelConfig()
FORMAT.validate()
KERNEL.validate()

RESULT_DIR = Path(
    'results/ppl/top2-dewa-fpacc-skip/llama2-7b'
)
ZIP_PATH = Path(
    'artifacts/colab-downloads/'
    'w-bfp4-a-bie4-top2cap-dewa-fpacc-excskip-t8-12-s2048-results.zip'
)
PRIVATE_VALUE_BITS = 1 + FORMAT.mantissa_bits
WEIGHT_BITS_PER_VALUE = (
    PRIVATE_VALUE_BITS + FORMAT.shared_exponent_bits / FORMAT.block_size
)
ACTIVATION_BITS_PER_VALUE = (
    PRIVATE_VALUE_BITS
    + 1
    + 2 * FORMAT.shared_exponent_bits / FORMAT.block_size
)

torch.manual_seed(0)
torch.backends.cuda.matmul.allow_tf32 = False

print(f'Hybrid config: {FORMAT}')
print(f'Kernel config: {KERNEL}')
print(f'DEWA thresholds: {T_SWEEP}')
print(f'Weight storage: {WEIGHT_BITS_PER_VALUE:.4f} bits/value')
print(f'Activation storage: {ACTIVATION_BITS_PER_VALUE:.4f} bits/value')
if not torch.cuda.is_available():
    print('CUDA is unavailable: CPU reference tests can run, but Triton/PPL cannot.')


## W-BFP4 and capped Top-2 A-BiE4 encoder

Weights use one shared exponent per Group-16 and never carry an outlier tag. Activations use the signed-tensor `mu + 3 sigma` candidate threshold, then retain at most the two largest-magnitude candidates in each Group-16. Every other candidate is demoted before the normal shared exponent is selected. Quantized values remain FP16 fake-quantized values, and activation type bits are packed little-endian along K.


In [ ]:
ACTIVATION_COUNT_NAMES = (
    "total_values",
    "candidate_values",
    "encoded_outlier_values",
    "demoted_values",
    "total_blocks",
    "candidate_affected_blocks",
    "encoded_affected_blocks",
    "cap_triggered_blocks",
    "encoded_normal_only_blocks",
    "encoded_outlier_only_blocks",
    "encoded_mixed_blocks",
)
WEIGHT_COUNT_NAMES = ("total_values", "total_blocks")


def _empty_activation_counts(device):
    return torch.zeros(
        len(ACTIVATION_COUNT_NAMES), dtype=torch.int64, device=device
    )


def _empty_weight_counts(device):
    return torch.zeros(
        len(WEIGHT_COUNT_NAMES), dtype=torch.int64, device=device
    )


def _rate(numerator, denominator):
    numerator = int(numerator)
    denominator = int(denominator)
    return {
        "numerator": numerator,
        "denominator": denominator,
        "rate": None if denominator == 0 else numerator / denominator,
    }


def _counts_to_dict(counts, names):
    values = counts.detach().cpu().tolist()
    return {name: int(value) for name, value in zip(names, values)}


def _histogram_percentile(histogram, q, first_bin=0):
    if not 0.0 <= q <= 1.0:
        raise ValueError("q must be in [0, 1].")
    total = sum(histogram[first_bin:])
    if total == 0:
        return None
    rank = max(1, math.ceil(q * total))
    cumulative = 0
    for value in range(first_bin, len(histogram)):
        cumulative += histogram[value]
        if cumulative >= rank:
            return value
    raise RuntimeError("Histogram percentile closure failed.")


def _summarize_activation(
    counts, candidate_histogram, encoded_histogram, include_tail
):
    candidate_histogram = [int(value) for value in candidate_histogram]
    encoded_histogram = [int(value) for value in encoded_histogram]
    total_blocks = counts["total_blocks"]
    candidate_values = counts["candidate_values"]
    encoded_values = counts["encoded_outlier_values"]
    demoted_values = counts["demoted_values"]
    candidate_affected = counts["candidate_affected_blocks"]
    encoded_affected = counts["encoded_affected_blocks"]
    cap_triggered = counts["cap_triggered_blocks"]

    expected_length = FORMAT.block_size + 1
    if len(candidate_histogram) != expected_length:
        raise RuntimeError("Unexpected candidate histogram length.")
    if len(encoded_histogram) != expected_length:
        raise RuntimeError("Unexpected encoded histogram length.")
    if sum(candidate_histogram) != total_blocks:
        raise RuntimeError("Candidate histogram does not close over blocks.")
    if sum(encoded_histogram) != total_blocks:
        raise RuntimeError("Encoded histogram does not close over blocks.")
    if sum(
        index * value for index, value in enumerate(candidate_histogram)
    ) != candidate_values:
        raise RuntimeError("Candidate histogram does not close over values.")
    if sum(
        index * value for index, value in enumerate(encoded_histogram)
    ) != encoded_values:
        raise RuntimeError("Encoded histogram does not close over values.")
    if candidate_values - encoded_values != demoted_values:
        raise RuntimeError("Candidate/encoded/demoted value closure failed.")
    if candidate_histogram[0] != total_blocks - candidate_affected:
        raise RuntimeError("Candidate zero-bin closure failed.")
    if sum(candidate_histogram[1:]) != candidate_affected:
        raise RuntimeError("Candidate affected-block closure failed.")
    if encoded_histogram[0] != counts["encoded_normal_only_blocks"]:
        raise RuntimeError("Encoded zero-bin closure failed.")
    if sum(encoded_histogram[1:]) != encoded_affected:
        raise RuntimeError("Encoded affected-block closure failed.")
    if candidate_affected != encoded_affected:
        raise RuntimeError("A nonempty candidate set must encode an outlier.")
    if sum(
        candidate_histogram[FORMAT.max_outliers_per_block + 1 :]
    ) != cap_triggered:
        raise RuntimeError("Cap-triggered block closure failed.")
    if sum(encoded_histogram[FORMAT.max_outliers_per_block + 1 :]) != 0:
        raise RuntimeError("Encoded occupancy exceeds Top-2.")
    if (
        counts["encoded_outlier_only_blocks"]
        + counts["encoded_mixed_blocks"]
        != encoded_affected
    ):
        raise RuntimeError("Encoded affected-block partition failed.")

    max_candidate = max(
        (index for index, value in enumerate(candidate_histogram) if value),
        default=None,
    )
    max_encoded = max(
        (index for index, value in enumerate(encoded_histogram) if value),
        default=None,
    )
    percentiles = (
        ("50", 0.50),
        ("90", 0.90),
        ("95", 0.95),
        ("99", 0.99),
        ("99_9", 0.999),
        ("99_99", 0.9999),
    )
    summary = {
        "counts": counts,
        "candidate_histogram_semantics": (
            "index k = blocks with exactly k threshold candidates before capping"
        ),
        "candidate_outliers_per_block_histogram": candidate_histogram,
        "encoded_histogram_semantics": (
            "index k = blocks with exactly k encoded outliers after Top-2 capping"
        ),
        "encoded_outliers_per_block_histogram": encoded_histogram,
        "max_candidate_outliers_per_block": max_candidate,
        "max_encoded_outliers_per_block": max_encoded,
        "nearest_rank_percentiles": {
            "candidate_all_blocks": {
                f"p{label}": _histogram_percentile(candidate_histogram, q)
                for label, q in percentiles
            },
            "candidate_affected_blocks_only": {
                f"p{label}": _histogram_percentile(
                    candidate_histogram, q, first_bin=1
                )
                for label, q in percentiles
            },
            "encoded_all_blocks": {
                f"p{label}": _histogram_percentile(encoded_histogram, q)
                for label, q in percentiles
            },
        },
        "rates": {
            "candidate_value_rate": _rate(
                candidate_values, counts["total_values"]
            ),
            "encoded_outlier_value_rate": _rate(
                encoded_values, counts["total_values"]
            ),
            "demoted_value_rate": _rate(
                demoted_values, counts["total_values"]
            ),
            "demoted_fraction_of_candidates": _rate(
                demoted_values, candidate_values
            ),
            "candidate_affected_block_rate": _rate(
                candidate_affected, total_blocks
            ),
            "encoded_affected_block_rate": _rate(
                encoded_affected, total_blocks
            ),
            "cap_triggered_block_rate": _rate(cap_triggered, total_blocks),
        },
    }
    if include_tail:
        summary["candidate_tail_probabilities"] = [
            {
                "minimum_candidates": minimum,
                **_rate(sum(candidate_histogram[minimum:]), total_blocks),
            }
            for minimum in range(1, FORMAT.block_size + 1)
        ]
    return summary


@torch.no_grad()
def _signed_tensor_threshold(tensor, sigma_k, chunk_rows):
    width = tensor.shape[-1]
    flat = tensor.reshape(-1, width)
    if flat.numel() == 0:
        raise ValueError("Cannot compute a threshold for an empty tensor.")
    if chunk_rows <= 0:
        raise ValueError("chunk_rows must be positive.")

    running_count = 0
    running_mean = torch.zeros((), dtype=torch.float64, device=flat.device)
    running_m2 = torch.zeros((), dtype=torch.float64, device=flat.device)

    for start in range(0, flat.size(0), chunk_rows):
        end = min(start + chunk_rows, flat.size(0))
        values = flat[start:end].float()
        chunk_count = values.numel()
        chunk_var, chunk_mean = torch.var_mean(values, unbiased=False)
        chunk_mean = chunk_mean.to(torch.float64)
        chunk_var = chunk_var.to(torch.float64)

        if running_count == 0:
            running_mean = chunk_mean
            running_m2 = chunk_var * chunk_count
            running_count = chunk_count
            continue

        combined_count = running_count + chunk_count
        delta = chunk_mean - running_mean
        running_mean = running_mean + delta * (chunk_count / combined_count)
        running_m2 = (
            running_m2
            + chunk_var * chunk_count
            + delta.square() * running_count * chunk_count / combined_count
        )
        running_count = combined_count

    variance = (running_m2 / running_count).clamp_min(0.0)
    threshold = running_mean + sigma_k * torch.sqrt(variance)
    return threshold.to(torch.float32)


def _shared_exponent(max_abs, present, config):
    safe_max = max_abs.clamp_min(torch.finfo(torch.float32).tiny)
    exponent = torch.floor(torch.log2(safe_max))
    exp_min = -(1 << (config.shared_exponent_bits - 1))
    exp_max = (1 << (config.shared_exponent_bits - 1)) - 1
    exponent = exponent.clamp(exp_min, exp_max)
    return torch.where(present, exponent, torch.zeros_like(exponent))


def _pack_bits_last_dim(mask):
    width = mask.shape[-1]
    padding = (-width) % 8
    if padding:
        mask = F.pad(mask, (0, padding), value=False)
    grouped = mask.reshape(*mask.shape[:-1], -1, 8).to(torch.int16)
    shifts = torch.arange(8, device=mask.device, dtype=torch.int16)
    shape = (1,) * (grouped.ndim - 1) + (8,)
    return (grouped << shifts.reshape(shape)).sum(dim=-1).to(torch.uint8)


def _unpack_bits_last_dim(packed, width):
    shifts = torch.arange(8, device=packed.device, dtype=torch.int16)
    shape = (1,) * packed.ndim + (8,)
    expanded = (
        (packed.to(torch.int16).unsqueeze(-1) >> shifts.reshape(shape)) & 1
    ).to(torch.bool)
    return expanded.reshape(*packed.shape[:-1], -1)[..., :width]


@torch.no_grad()
def _quantize_weight_bfp_rows(rows, config):
    original_shape = rows.shape
    width = original_shape[-1]
    flat = rows.reshape(-1, width).float()
    padding = (-width) % config.block_size
    if padding:
        flat = F.pad(flat, (0, padding))

    padded_width = flat.size(1)
    blocks = flat.reshape(flat.size(0), -1, config.block_size)
    max_abs = blocks.abs().amax(dim=-1, keepdim=True)
    exponent = _shared_exponent(max_abs, max_abs != 0, config)
    step = torch.pow(2.0, exponent - (config.mantissa_bits - 1))
    mantissa_max = (1 << config.mantissa_bits) - 1
    mantissa = torch.round(blocks / step).clamp(
        -mantissa_max, mantissa_max
    )
    output = (mantissa * step).reshape(flat.size(0), padded_width)
    return output[:, :width].reshape(original_shape).to(rows.dtype)


@torch.no_grad()
def quantize_weight_in_place(weight, config):
    counts = _empty_weight_counts(weight.device)
    blocks_per_row = math.ceil(weight.size(-1) / config.block_size)

    for start in range(0, weight.size(0), config.weight_chunk_rows):
        end = min(start + config.weight_chunk_rows, weight.size(0))
        rows = weight[start:end]
        rows.copy_(_quantize_weight_bfp_rows(rows, config))
        chunk_counts = torch.tensor(
            [rows.numel(), rows.size(0) * blocks_per_row],
            dtype=torch.int64,
            device=weight.device,
        )
        counts.add_(chunk_counts)
    return counts


def _select_top2_candidates(magnitude, candidate, config):
    if magnitude.shape != candidate.shape:
        raise ValueError("Magnitude and candidate masks must have the same shape.")
    if magnitude.size(-1) != config.block_size:
        raise ValueError("Top-2 selection expects G16 in the last dimension.")

    selected = torch.zeros_like(candidate)
    remaining = candidate.clone()
    positions = torch.arange(config.block_size, device=magnitude.device)
    positions = positions.view(
        *([1] * (magnitude.ndim - 1)), config.block_size
    )

    for _ in range(config.max_outliers_per_block):
        has_candidate = remaining.any(dim=-1, keepdim=True)
        score = magnitude.masked_fill(~remaining, float("-inf"))
        index = score.argmax(dim=-1, keepdim=True)
        picked = (positions == index) & has_candidate
        selected = selected | picked
        remaining = remaining & ~picked
    return selected


@torch.no_grad()
def _quantize_activation_rows_with_threshold(rows, threshold, config):
    original_shape = rows.shape
    width = original_shape[-1]
    flat = rows.reshape(-1, width).float()
    padding = (-width) % config.block_size

    valid = torch.ones_like(flat, dtype=torch.bool)
    if padding:
        flat = F.pad(flat, (0, padding))
        valid = F.pad(valid, (0, padding), value=False)

    padded_width = flat.size(1)
    blocks = flat.reshape(flat.size(0), -1, config.block_size)
    valid_blocks = valid.reshape_as(blocks)
    magnitude = blocks.abs()

    candidate = valid_blocks & (magnitude > threshold)
    outlier = _select_top2_candidates(magnitude, candidate, config)
    demoted = candidate & ~outlier
    normal = valid_blocks & ~outlier
    normal_present = normal.any(dim=-1, keepdim=True)
    outlier_present = outlier.any(dim=-1, keepdim=True)

    normal_max = torch.where(normal, magnitude, 0.0).amax(dim=-1, keepdim=True)
    outlier_max = torch.where(outlier, magnitude, 0.0).amax(dim=-1, keepdim=True)
    normal_exp = _shared_exponent(normal_max, normal_present, config)
    outlier_exp = _shared_exponent(outlier_max, outlier_present, config)
    normal_exp = torch.where(
        ~normal_present & outlier_present, outlier_exp, normal_exp
    )
    outlier_exp = torch.where(
        ~outlier_present & normal_present, normal_exp, outlier_exp
    )
    selected_exp = torch.where(outlier, outlier_exp, normal_exp)

    step = torch.pow(2.0, selected_exp - (config.mantissa_bits - 1))
    mantissa_max = (1 << config.mantissa_bits) - 1
    mantissa = torch.round(blocks / step).clamp(
        -mantissa_max, mantissa_max
    )
    dequantized = (mantissa * step).reshape(flat.size(0), padded_width)
    dequantized = dequantized[:, :width].reshape(original_shape).to(rows.dtype)
    outlier_mask = outlier.reshape(flat.size(0), padded_width)[:, :width]

    real_block = valid_blocks.any(dim=-1)
    valid_count = valid_blocks.sum(dim=-1)
    candidate_count = candidate.sum(dim=-1)
    outlier_count = outlier.sum(dim=-1)
    candidate_affected = real_block & (candidate_count > 0)
    encoded_affected = real_block & (outlier_count > 0)
    cap_triggered = real_block & (
        candidate_count > config.max_outliers_per_block
    )
    outlier_only = real_block & (outlier_count == valid_count)
    mixed = encoded_affected & ~outlier_only
    candidate_histogram = torch.bincount(
        candidate_count[real_block], minlength=config.block_size + 1
    )
    encoded_histogram = torch.bincount(
        outlier_count[real_block], minlength=config.block_size + 1
    )
    counts = torch.stack(
        (
            torch.tensor(rows.numel(), dtype=torch.int64, device=rows.device),
            candidate.sum(dtype=torch.int64),
            outlier.sum(dtype=torch.int64),
            demoted.sum(dtype=torch.int64),
            real_block.sum(dtype=torch.int64),
            candidate_affected.sum(dtype=torch.int64),
            encoded_affected.sum(dtype=torch.int64),
            cap_triggered.sum(dtype=torch.int64),
            (real_block & ~encoded_affected).sum(dtype=torch.int64),
            outlier_only.sum(dtype=torch.int64),
            mixed.sum(dtype=torch.int64),
        )
    )
    return (
        dequantized,
        outlier_mask,
        counts,
        candidate_histogram,
        encoded_histogram,
    )


@torch.no_grad()
def quantize_activation_top2(tensor, config, chunk_rows):
    width = tensor.shape[-1]
    flat = tensor.reshape(-1, width)
    threshold = _signed_tensor_threshold(flat, config.sigma_k, chunk_rows)
    output = torch.empty_like(flat)
    packed_tags = torch.empty(
        (flat.size(0), math.ceil(width / 8)),
        dtype=torch.uint8,
        device=flat.device,
    )
    counts = _empty_activation_counts(flat.device)
    candidate_histogram = torch.zeros(
        config.block_size + 1, dtype=torch.int64, device=flat.device
    )
    encoded_histogram = torch.zeros(
        config.block_size + 1, dtype=torch.int64, device=flat.device
    )
    outlier_per_k = torch.zeros(
        width, dtype=torch.int64, device=flat.device
    )
    outlier_groups = torch.zeros(
        math.ceil(width / config.block_size),
        dtype=torch.int64,
        device=flat.device,
    )

    for start in range(0, flat.size(0), chunk_rows):
        end = min(start + chunk_rows, flat.size(0))
        (
            quantized,
            outlier_mask,
            chunk_counts,
            chunk_candidate_histogram,
            chunk_encoded_histogram,
        ) = _quantize_activation_rows_with_threshold(
            flat[start:end], threshold, config
        )
        output[start:end] = quantized
        packed_tags[start:end] = _pack_bits_last_dim(outlier_mask)
        counts.add_(chunk_counts)
        candidate_histogram.add_(chunk_candidate_histogram)
        encoded_histogram.add_(chunk_encoded_histogram)
        outlier_per_k.add_(
            outlier_mask.sum(dim=0, dtype=torch.int64)
        )

        group_padding = (-width) % config.block_size
        grouped_mask = outlier_mask
        if group_padding:
            grouped_mask = F.pad(grouped_mask, (0, group_padding), value=False)
        outlier_groups.add_(
            grouped_mask.reshape(
                grouped_mask.size(0), -1, config.block_size
            ).any(dim=-1).sum(dim=0, dtype=torch.int64)
        )

    routing_meta = {
        "rows": flat.size(0),
        "width": width,
        "outlier_per_k": outlier_per_k,
        "outlier_groups": outlier_groups,
    }
    return (
        output.reshape_as(tensor),
        packed_tags,
        threshold,
        counts,
        candidate_histogram,
        encoded_histogram,
        routing_meta,
    )


## Hybrid product partition and PyTorch reference

For every Group-16 and output element, products form exactly two partial sums:

- `P_N`: all products whose activation is encoded as normal.
- `P_O`: the zero, one, or two products whose activation is encoded as outlier.

Both decisions compare against the same pre-update numeric DEWA accumulator `A_old`. `P_N` follows the symmetric DEWA skip/replace/add rule. Nonzero `P_O` uses a one-sided admission rule: skip only when `E(P_O) - E(A_old) <= -T_exc`; otherwise send it to the FP32 exception accumulator. When `A_old == 0`, a nonzero `P_O` is always sent.


In [ ]:
KERNEL_STAT_NAMES = (
    "total_group_output_slots",
    "normal_initial_load",
    "zero_normal_partial",
    "normal_skip_new",
    "normal_replace_old",
    "normal_add",
    "normal_nonzero_decisions",
    "exception_routed_group_slots",
    "zero_exception_partial",
    "nonzero_exception_partial",
    "exception_send_no_reference",
    "exception_skip_small",
    "exception_send_referenced",
    "exception_fp_acc_adds",
    "final_dewa_nonzero_flushes",
)

PRODUCT_STAT_NAMES = (
    "total_products",
    "normal_products",
    "exception_products",
    "expected_total_group_output_slots",
    "expected_exception_routed_group_slots",
    "expected_output_slots",
)


def _empty_kernel_stats(device):
    return torch.zeros(
        len(KERNEL_STAT_NAMES), dtype=torch.int64, device=device
    )


def _empty_product_stats(device):
    return torch.zeros(
        len(PRODUCT_STAT_NAMES), dtype=torch.int64, device=device
    )


def _named_stats(tensor, names):
    values = tensor.detach().cpu().tolist()
    return {name: int(value) for name, value in zip(names, values)}


def _activation_product_partition_tensor(activation_meta, output_rows):
    activation_rows = int(activation_meta["rows"])
    width = int(activation_meta["width"])
    outlier_per_k = activation_meta["outlier_per_k"]
    exception = (outlier_per_k * output_rows).sum()
    total = torch.tensor(
        activation_rows * output_rows * width,
        dtype=torch.int64,
        device=outlier_per_k.device,
    )
    normal = total - exception
    return total, normal, exception


def _expected_slot_tensors(activation_meta, output_rows):
    activation_rows = int(activation_meta["rows"])
    outlier_groups = activation_meta["outlier_groups"]
    total_group_slots = torch.tensor(
        outlier_groups.numel() * activation_rows * output_rows,
        dtype=torch.int64,
        device=outlier_groups.device,
    )
    routed_group_slots = (outlier_groups * output_rows).sum()
    output_slots = torch.tensor(
        activation_rows * output_rows,
        dtype=torch.int64,
        device=outlier_groups.device,
    )
    return total_group_slots, routed_group_slots, output_slots


def _leading_exponent(values):
    nonzero = values != 0
    exponent = torch.floor(
        torch.log2(
            values.abs().clamp_min(torch.finfo(torch.float32).tiny)
        )
    )
    return exponent, nonzero


def _dewa_reference_update(accumulator, partial, threshold_bits, enabled):
    old_exponent, old_nonzero = _leading_exponent(accumulator)
    new_exponent, new_nonzero = _leading_exponent(partial)
    both = old_nonzero & new_nonzero
    delta = new_exponent - old_exponent

    skip_new = both & enabled & (delta <= -threshold_bits)
    replace_old = both & enabled & (delta >= threshold_bits)
    normal_add = both & ~skip_new & ~replace_old
    load = ~old_nonzero & new_nonzero

    updated = torch.where(load | replace_old, partial, accumulator)
    updated = torch.where(normal_add, accumulator + partial, updated)
    counts = {
        "normal_initial_load": load.sum(dtype=torch.int64),
        "zero_normal_partial": (~new_nonzero).sum(dtype=torch.int64),
        "normal_skip_new": skip_new.sum(dtype=torch.int64),
        "normal_replace_old": replace_old.sum(dtype=torch.int64),
        "normal_add": normal_add.sum(dtype=torch.int64),
        "normal_nonzero_decisions": both.sum(dtype=torch.int64),
    }
    return updated, counts


def _exception_reference_decision(
    pre_update_accumulator,
    exception_partial,
    threshold_bits,
    enabled,
):
    old_exponent, old_nonzero = _leading_exponent(pre_update_accumulator)
    exception_exponent, exception_nonzero = _leading_exponent(
        exception_partial
    )
    has_reference = old_nonzero & exception_nonzero
    delta = exception_exponent - old_exponent

    skip_small = (
        has_reference & enabled & (delta <= -threshold_bits)
    )
    send_no_reference = ~old_nonzero & exception_nonzero
    send_referenced = has_reference & ~skip_small
    send = send_no_reference | send_referenced
    return send, {
        "exception_send_no_reference": send_no_reference,
        "exception_skip_small": skip_small,
        "exception_send_referenced": send_referenced,
        "exception_nonzero": exception_nonzero,
    }


@torch.no_grad()
def reference_hybrid_linear(
    x,
    x_outlier,
    weight,
    bias,
    dewa_config,
    group_k=16,
):
    if x.ndim != 2 or weight.ndim != 2:
        raise ValueError("Reference expects flattened 2-D operands.")
    m, k = x.shape
    n, weight_k = weight.shape
    if k != weight_k or k % group_k:
        raise ValueError("K must match and be divisible by group_k.")
    dewa_config.validate()

    normal_acc = torch.zeros((m, n), dtype=torch.float32, device=x.device)
    fp_acc = torch.zeros_like(normal_acc)
    stats = _empty_kernel_stats(x.device)

    for start in range(0, k, group_k):
        end = start + group_k
        x_group = x[:, start:end].float()
        w_group = weight[:, start:end].float()
        x_tag = x_outlier[:, start:end]

        x_normal = torch.where(x_tag, 0.0, x_group)
        x_exception = torch.where(x_tag, x_group, 0.0)
        normal_partial = x_normal @ w_group.transpose(0, 1)
        exception_partial = x_exception @ w_group.transpose(0, 1)

        pre_update_acc = normal_acc
        send_exception, exception_decision = (
            _exception_reference_decision(
                pre_update_acc,
                exception_partial,
                dewa_config.exception_threshold_bits,
                dewa_config.exception_skip_enabled,
            )
        )
        normal_acc, normal_counts = _dewa_reference_update(
            pre_update_acc,
            normal_partial,
            dewa_config.threshold_bits,
            dewa_config.enabled,
        )
        fp_acc = torch.where(
            send_exception, fp_acc + exception_partial, fp_acc
        )

        routed = x_tag.any(dim=1, keepdim=True).expand(m, n)
        exception_nonzero = exception_decision["exception_nonzero"]
        stats[0].add_(m * n)
        for index, name in enumerate(KERNEL_STAT_NAMES[1:7], start=1):
            stats[index].add_(normal_counts[name])
        stats[7].add_(routed.sum(dtype=torch.int64))
        stats[8].add_(
            (routed & ~exception_nonzero).sum(dtype=torch.int64)
        )
        stats[9].add_(
            (routed & exception_nonzero).sum(dtype=torch.int64)
        )
        stats[10].add_(
            exception_decision["exception_send_no_reference"].sum(
                dtype=torch.int64
            )
        )
        stats[11].add_(
            exception_decision["exception_skip_small"].sum(
                dtype=torch.int64
            )
        )
        stats[12].add_(
            exception_decision["exception_send_referenced"].sum(
                dtype=torch.int64
            )
        )
        stats[13].add_(send_exception.sum(dtype=torch.int64))

    stats[14].add_((normal_acc != 0).sum(dtype=torch.int64))
    output = normal_acc + fp_acc
    if bias is not None:
        output = output + bias.float().unsqueeze(0)
    return output.to(torch.float16), stats


## Fused Top-2 DEWA + conditional FP-ACC Triton kernel

Each K-group computes `P_N` and `P_O` independently. The exception decision is made before `P_N` changes the normal accumulator, so both paths observe exactly the same `A_old`:

- normal path: load, skip, replace, or align-and-add in DEWA;
- exception path: zero/no-request, no-reference send, low-side skip, or send to FP32;
- output: `FP16(normal_acc + exception_fp_acc + bias)`.


In [ ]:
@triton.jit
def _top2_dewa_fpacc_kernel(
    x_ptr,
    x_tag_ptr,
    w_ptr,
    bias_ptr,
    y_ptr,
    stats_ptr,
    M,
    N,
    K,
    stride_xm,
    stride_xk,
    stride_xtm,
    stride_xtb,
    stride_wn,
    stride_wk,
    stride_ym,
    stride_yn,
    HAS_BIAS: tl.constexpr,
    DEWA_ENABLED: tl.constexpr,
    EXCEPTION_SKIP_ENABLED: tl.constexpr,
    THRESHOLD_BITS: tl.constexpr,
    EXCEPTION_THRESHOLD_BITS: tl.constexpr,
    GROUP_K: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    COLLECT_STATS: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, GROUP_K)
    tag_bytes = offs_k // 8
    tag_bits = offs_k % 8

    x_ptrs = (
        x_ptr
        + offs_m[:, None] * stride_xm
        + offs_k[None, :] * stride_xk
    )
    w_ptrs = (
        w_ptr
        + offs_n[:, None] * stride_wn
        + offs_k[None, :] * stride_wk
    )
    x_tag_ptrs = (
        x_tag_ptr
        + offs_m[:, None] * stride_xtm
        + tag_bytes[None, :] * stride_xtb
    )

    valid_m = offs_m < M
    valid_n = offs_n < N
    valid_output = valid_m[:, None] & valid_n[None, :]

    normal_acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    fp_acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    stat_total_slots = tl.zeros((1,), dtype=tl.int32)
    stat_normal_load = tl.zeros((1,), dtype=tl.int32)
    stat_zero_normal = tl.zeros((1,), dtype=tl.int32)
    stat_normal_skip = tl.zeros((1,), dtype=tl.int32)
    stat_normal_replace = tl.zeros((1,), dtype=tl.int32)
    stat_normal_add = tl.zeros((1,), dtype=tl.int32)
    stat_normal_decisions = tl.zeros((1,), dtype=tl.int32)
    stat_exception_routed = tl.zeros((1,), dtype=tl.int32)
    stat_zero_exception = tl.zeros((1,), dtype=tl.int32)
    stat_nonzero_exception = tl.zeros((1,), dtype=tl.int32)
    stat_exception_no_ref = tl.zeros((1,), dtype=tl.int32)
    stat_exception_skip = tl.zeros((1,), dtype=tl.int32)
    stat_exception_ref_send = tl.zeros((1,), dtype=tl.int32)
    stat_exception_fp_add = tl.zeros((1,), dtype=tl.int32)

    for _ in range(0, tl.cdiv(K, GROUP_K)):
        x_block = tl.load(
            x_ptrs, mask=valid_m[:, None], other=0.0
        )
        w_block = tl.load(
            w_ptrs, mask=valid_n[:, None], other=0.0
        )
        x_tag_word = tl.load(
            x_tag_ptrs, mask=valid_m[:, None], other=0
        ).to(tl.int32)
        x_outlier = ((x_tag_word >> tag_bits[None, :]) & 1) != 0

        x_zero = tl.zeros_like(x_block)
        x_normal = tl.where(x_outlier, x_zero, x_block)
        x_exception = tl.where(x_outlier, x_block, x_zero)
        normal_partial = tl.dot(
            x_normal, tl.trans(w_block)
        ).to(tl.float32)
        exception_partial = tl.dot(
            x_exception, tl.trans(w_block)
        ).to(tl.float32)

        # Both decisions use the same pre-update normal accumulator state.
        old_nonzero = normal_acc != 0.0
        old_abs = tl.abs(normal_acc)
        old_exp = tl.floor(
            tl.log2(tl.where(old_nonzero, old_abs, 1.0))
        )

        new_nonzero = normal_partial != 0.0
        new_abs = tl.abs(normal_partial)
        new_exp = tl.floor(
            tl.log2(tl.where(new_nonzero, new_abs, 1.0))
        )
        normal_both = valid_output & old_nonzero & new_nonzero
        normal_delta = new_exp - old_exp
        normal_skip = (
            normal_both
            & DEWA_ENABLED
            & (normal_delta <= -THRESHOLD_BITS)
        )
        normal_replace = (
            normal_both
            & DEWA_ENABLED
            & (normal_delta >= THRESHOLD_BITS)
        )
        normal_add = normal_both & ~normal_skip & ~normal_replace
        normal_load = valid_output & ~old_nonzero & new_nonzero

        exception_nonzero = exception_partial != 0.0
        exception_abs = tl.abs(exception_partial)
        exception_exp = tl.floor(
            tl.log2(
                tl.where(exception_nonzero, exception_abs, 1.0)
            )
        )
        exception_has_ref = (
            valid_output & old_nonzero & exception_nonzero
        )
        exception_delta = exception_exp - old_exp
        exception_skip = (
            exception_has_ref
            & EXCEPTION_SKIP_ENABLED
            & (
                exception_delta
                <= -EXCEPTION_THRESHOLD_BITS
            )
        )
        exception_send_no_ref = (
            valid_output & ~old_nonzero & exception_nonzero
        )
        exception_send_ref = exception_has_ref & ~exception_skip
        exception_send = exception_send_no_ref | exception_send_ref

        fp_acc = tl.where(
            exception_send,
            fp_acc + exception_partial,
            fp_acc,
        )
        normal_updated = tl.where(
            normal_load | normal_replace,
            normal_partial,
            normal_acc,
        )
        normal_acc = tl.where(
            normal_add,
            normal_acc + normal_partial,
            normal_updated,
        )

        if COLLECT_STATS:
            x_group_outlier = (
                tl.sum(x_outlier.to(tl.int32), axis=1) > 0
            )
            exception_routed = (
                valid_output & x_group_outlier[:, None]
            )
            zero_exception = (
                exception_routed & ~exception_nonzero
            )
            nonzero_exception = (
                exception_routed & exception_nonzero
            )

            stat_total_slots += tl.sum(
                tl.sum(valid_output.to(tl.int32), axis=1), axis=0
            )
            stat_normal_load += tl.sum(
                tl.sum(normal_load.to(tl.int32), axis=1), axis=0
            )
            stat_zero_normal += tl.sum(
                tl.sum(
                    (
                        valid_output & ~new_nonzero
                    ).to(tl.int32),
                    axis=1,
                ),
                axis=0,
            )
            stat_normal_skip += tl.sum(
                tl.sum(normal_skip.to(tl.int32), axis=1), axis=0
            )
            stat_normal_replace += tl.sum(
                tl.sum(normal_replace.to(tl.int32), axis=1), axis=0
            )
            stat_normal_add += tl.sum(
                tl.sum(normal_add.to(tl.int32), axis=1), axis=0
            )
            stat_normal_decisions += tl.sum(
                tl.sum(normal_both.to(tl.int32), axis=1), axis=0
            )
            stat_exception_routed += tl.sum(
                tl.sum(exception_routed.to(tl.int32), axis=1),
                axis=0,
            )
            stat_zero_exception += tl.sum(
                tl.sum(zero_exception.to(tl.int32), axis=1),
                axis=0,
            )
            stat_nonzero_exception += tl.sum(
                tl.sum(nonzero_exception.to(tl.int32), axis=1),
                axis=0,
            )
            stat_exception_no_ref += tl.sum(
                tl.sum(exception_send_no_ref.to(tl.int32), axis=1),
                axis=0,
            )
            stat_exception_skip += tl.sum(
                tl.sum(exception_skip.to(tl.int32), axis=1),
                axis=0,
            )
            stat_exception_ref_send += tl.sum(
                tl.sum(exception_send_ref.to(tl.int32), axis=1),
                axis=0,
            )
            stat_exception_fp_add += tl.sum(
                tl.sum(exception_send.to(tl.int32), axis=1),
                axis=0,
            )

        x_ptrs += GROUP_K * stride_xk
        w_ptrs += GROUP_K * stride_wk
        x_tag_ptrs += (GROUP_K // 8) * stride_xtb

    output = normal_acc + fp_acc
    if HAS_BIAS:
        bias = tl.load(
            bias_ptr + offs_n, mask=valid_n, other=0.0
        )
        output += bias[None, :]

    y_ptrs = (
        y_ptr
        + offs_m[:, None] * stride_ym
        + offs_n[None, :] * stride_yn
    )
    tl.store(y_ptrs, output, mask=valid_output)

    if COLLECT_STATS:
        stat_final_flush = tl.sum(
            tl.sum(
                (
                    valid_output & (normal_acc != 0.0)
                ).to(tl.int32),
                axis=1,
            ),
            axis=0,
        )
        tl.atomic_add(
            stats_ptr + 0,
            tl.sum(stat_total_slots, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 1,
            tl.sum(stat_normal_load, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 2,
            tl.sum(stat_zero_normal, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 3,
            tl.sum(stat_normal_skip, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 4,
            tl.sum(stat_normal_replace, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 5,
            tl.sum(stat_normal_add, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 6,
            tl.sum(stat_normal_decisions, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 7,
            tl.sum(stat_exception_routed, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 8,
            tl.sum(stat_zero_exception, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 9,
            tl.sum(stat_nonzero_exception, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 10,
            tl.sum(stat_exception_no_ref, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 11,
            tl.sum(stat_exception_skip, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 12,
            tl.sum(stat_exception_ref_send, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 13,
            tl.sum(stat_exception_fp_add, axis=0).to(tl.int64),
        )
        tl.atomic_add(stats_ptr + 14, stat_final_flush.to(tl.int64))


def fused_top2_dewa_linear(
    x,
    x_packed_tags,
    weight,
    bias,
    dewa_config,
    kernel_stats,
):
    dewa_config.validate()
    original_shape = x.shape
    k = original_shape[-1]
    x_2d = x.reshape(-1, k).contiguous()
    m = x_2d.size(0)
    n = weight.size(0)

    if k != weight.size(1):
        raise ValueError("Activation and weight K dimensions differ.")
    if k % KERNEL.group_k or k % 8:
        raise ValueError("K must be divisible by Group-16 and tag width.")
    if x_packed_tags.shape != (m, k // 8):
        raise ValueError("Unexpected activation tag shape.")
    if kernel_stats.numel() != len(KERNEL_STAT_NAMES):
        raise ValueError("Unexpected kernel statistics shape.")

    output = torch.empty(
        (m, n), dtype=torch.float16, device=x.device
    )
    bias_arg = bias if bias is not None else weight
    grid = (
        triton.cdiv(m, KERNEL.block_m),
        triton.cdiv(n, KERNEL.block_n),
    )
    _top2_dewa_fpacc_kernel[grid](
        x_2d,
        x_packed_tags,
        weight,
        bias_arg,
        output,
        kernel_stats,
        M=m,
        N=n,
        K=k,
        stride_xm=x_2d.stride(0),
        stride_xk=x_2d.stride(1),
        stride_xtm=x_packed_tags.stride(0),
        stride_xtb=x_packed_tags.stride(1),
        stride_wn=weight.stride(0),
        stride_wk=weight.stride(1),
        stride_ym=output.stride(0),
        stride_yn=output.stride(1),
        HAS_BIAS=bias is not None,
        DEWA_ENABLED=dewa_config.enabled,
        EXCEPTION_SKIP_ENABLED=dewa_config.exception_skip_enabled,
        THRESHOLD_BITS=dewa_config.threshold_bits,
        EXCEPTION_THRESHOLD_BITS=(
            dewa_config.exception_threshold_bits
        ),
        GROUP_K=KERNEL.group_k,
        BLOCK_M=KERNEL.block_m,
        BLOCK_N=KERNEL.block_n,
        COLLECT_STATS=True,
        num_warps=KERNEL.num_warps,
        num_stages=KERNEL.num_stages,
    )
    return output.reshape(*original_shape[:-1], n)


## Linear replacement and self-describing statistics


In [ ]:
class Top2DEWAFPACCLinear(nn.Module):
    def __init__(
        self,
        linear,
        format_config,
        dewa_config,
        weight_counts,
    ):
        super().__init__()
        self.linear = linear
        self.format_config = format_config
        self.dewa_config = dewa_config
        self.weight_counts = _counts_to_dict(
            weight_counts, WEIGHT_COUNT_NAMES
        )

        device = linear.weight.device
        self.register_buffer(
            "_activation_counts",
            _empty_activation_counts(device),
            persistent=False,
        )
        self.register_buffer(
            "_candidate_histogram",
            torch.zeros(
                format_config.block_size + 1,
                dtype=torch.int64,
                device=device,
            ),
            persistent=False,
        )
        self.register_buffer(
            "_encoded_histogram",
            torch.zeros(
                format_config.block_size + 1,
                dtype=torch.int64,
                device=device,
            ),
            persistent=False,
        )
        self.register_buffer(
            "_activation_calls",
            torch.zeros((), dtype=torch.int64, device=device),
            persistent=False,
        )
        self.register_buffer(
            "_activation_threshold_sum",
            torch.zeros((), dtype=torch.float64, device=device),
            persistent=False,
        )
        self.register_buffer(
            "_activation_threshold_min",
            torch.tensor(
                float("inf"), dtype=torch.float64, device=device
            ),
            persistent=False,
        )
        self.register_buffer(
            "_activation_threshold_max",
            torch.tensor(
                float("-inf"), dtype=torch.float64, device=device
            ),
            persistent=False,
        )
        self.register_buffer(
            "_kernel_stats",
            _empty_kernel_stats(device),
            persistent=False,
        )
        self.register_buffer(
            "_product_stats",
            _empty_product_stats(device),
            persistent=False,
        )

    def reset_runtime_stats(self):
        self._activation_counts.zero_()
        self._candidate_histogram.zero_()
        self._encoded_histogram.zero_()
        self._activation_calls.zero_()
        self._activation_threshold_sum.zero_()
        self._activation_threshold_min.fill_(float("inf"))
        self._activation_threshold_max.fill_(float("-inf"))
        self._kernel_stats.zero_()
        self._product_stats.zero_()

    def set_dewa_config(self, config):
        config.validate()
        self.dewa_config = config

    def forward(self, x):
        (
            x_quantized,
            x_packed_tags,
            threshold,
            activation_counts,
            candidate_histogram,
            encoded_histogram,
            activation_meta,
        ) = quantize_activation_top2(
            x,
            self.format_config,
            self.format_config.activation_chunk_rows,
        )

        threshold64 = threshold.to(torch.float64)
        self._activation_counts.add_(activation_counts)
        self._candidate_histogram.add_(candidate_histogram)
        self._encoded_histogram.add_(encoded_histogram)
        self._activation_calls.add_(1)
        self._activation_threshold_sum.add_(threshold64)
        self._activation_threshold_min.copy_(
            torch.minimum(self._activation_threshold_min, threshold64)
        )
        self._activation_threshold_max.copy_(
            torch.maximum(self._activation_threshold_max, threshold64)
        )

        partition = _activation_product_partition_tensor(
            activation_meta, self.linear.out_features
        )
        slots = _expected_slot_tensors(
            activation_meta, self.linear.out_features
        )
        self._product_stats.add_(torch.stack((*partition, *slots)))

        return fused_top2_dewa_linear(
            x_quantized,
            x_packed_tags,
            self.linear.weight,
            self.linear.bias,
            self.dewa_config,
            self._kernel_stats,
        )

    def export_stats(self):
        calls = int(self._activation_calls.detach().cpu().item())
        if calls:
            threshold_summary = {
                "count": calls,
                "mean": float(
                    self._activation_threshold_sum.detach().cpu().item()
                    / calls
                ),
                "min": float(
                    self._activation_threshold_min.detach().cpu().item()
                ),
                "max": float(
                    self._activation_threshold_max.detach().cpu().item()
                ),
            }
        else:
            threshold_summary = {
                "count": 0,
                "mean": None,
                "min": None,
                "max": None,
            }
        activation_counts = _counts_to_dict(
            self._activation_counts, ACTIVATION_COUNT_NAMES
        )
        activation_summary = _summarize_activation(
            activation_counts,
            self._candidate_histogram.detach().cpu().tolist(),
            self._encoded_histogram.detach().cpu().tolist(),
            include_tail=False,
        )
        return {
            "weight": {"counts": self.weight_counts},
            "activation": {
                "threshold_summary": threshold_summary,
                **activation_summary,
            },
            "kernel_counts": _named_stats(
                self._kernel_stats, KERNEL_STAT_NAMES
            ),
            "product_counts": _named_stats(
                self._product_stats, PRODUCT_STAT_NAMES
            ),
        }


def replace_linear_layers(
    module, format_config, dewa_config, prefix=""
):
    replaced = {}
    for name, child in list(module.named_children()):
        full_name = f"{prefix}.{name}" if prefix else name
        if isinstance(child, nn.Linear):
            if (
                full_name == "lm_head"
                and not format_config.quantize_lm_head
            ):
                continue
            if child.in_features % format_config.block_size:
                raise ValueError(
                    f"{full_name}: K must be divisible by G16."
                )
            weight_counts = quantize_weight_in_place(
                child.weight, format_config
            )
            wrapper = Top2DEWAFPACCLinear(
                child,
                format_config,
                dewa_config,
                weight_counts,
            )
            setattr(module, name, wrapper)
            replaced[full_name] = wrapper
        else:
            replaced.update(
                replace_linear_layers(
                    child,
                    format_config,
                    dewa_config,
                    full_name,
                )
            )
    return replaced


def configure_dewa(replaced, threshold_bits):
    config = DEWAConfig(
        threshold_bits=threshold_bits,
        exception_threshold_bits=threshold_bits,
        enabled=True,
        exception_skip_enabled=True,
    )
    config.validate()
    for layer in replaced.values():
        layer.set_dewa_config(config)
        layer.reset_runtime_stats()
    return config


def _add_named_counts(total, current):
    for name in total:
        total[name] += current[name]


def export_experiment_stats(replaced):
    weight_totals = {name: 0 for name in WEIGHT_COUNT_NAMES}
    activation_totals = {
        name: 0 for name in ACTIVATION_COUNT_NAMES
    }
    kernel_totals = {name: 0 for name in KERNEL_STAT_NAMES}
    product_totals = {name: 0 for name in PRODUCT_STAT_NAMES}
    candidate_histogram = [0] * (FORMAT.block_size + 1)
    encoded_histogram = [0] * (FORMAT.block_size + 1)
    threshold_sum = 0.0
    threshold_count = 0
    threshold_min = float("inf")
    threshold_max = float("-inf")
    layers = []

    for name in sorted(replaced):
        stats = replaced[name].export_stats()
        layers.append({"layer_name": name, **stats})
        _add_named_counts(weight_totals, stats["weight"]["counts"])
        _add_named_counts(
            activation_totals, stats["activation"]["counts"]
        )
        _add_named_counts(kernel_totals, stats["kernel_counts"])
        _add_named_counts(product_totals, stats["product_counts"])

        for index, value in enumerate(
            stats["activation"][
                "candidate_outliers_per_block_histogram"
            ]
        ):
            candidate_histogram[index] += value
        for index, value in enumerate(
            stats["activation"][
                "encoded_outliers_per_block_histogram"
            ]
        ):
            encoded_histogram[index] += value

        threshold = stats["activation"]["threshold_summary"]
        if threshold["count"]:
            threshold_sum += threshold["mean"] * threshold["count"]
            threshold_count += threshold["count"]
            threshold_min = min(threshold_min, threshold["min"])
            threshold_max = max(threshold_max, threshold["max"])

    threshold_summary = {
        "count": threshold_count,
        "mean": (
            None
            if threshold_count == 0
            else threshold_sum / threshold_count
        ),
        "min": None if threshold_count == 0 else threshold_min,
        "max": None if threshold_count == 0 else threshold_max,
    }
    activation_summary = _summarize_activation(
        activation_totals,
        candidate_histogram,
        encoded_histogram,
        include_tail=True,
    )

    decisions = kernel_totals["normal_nonzero_decisions"]
    normal_oow = (
        kernel_totals["normal_skip_new"]
        + kernel_totals["normal_replace_old"]
    )
    total_slots = kernel_totals["total_group_output_slots"]
    routed_slots = kernel_totals["exception_routed_group_slots"]
    nonzero_exception = kernel_totals["nonzero_exception_partial"]
    fp_adds = kernel_totals["exception_fp_acc_adds"]
    final_flushes = kernel_totals["final_dewa_nonzero_flushes"]
    total_fp_requests = fp_adds + final_flushes

    aggregate = {
        "weight": {
            "counts": weight_totals,
            "shared_exponents_per_block": 1,
        },
        "activation": {
            "threshold_summary": threshold_summary,
            **activation_summary,
        },
        "kernel_counts": kernel_totals,
        "product_counts": product_totals,
        "rates": {
            "dewa_oow_rate": _rate(normal_oow, decisions),
            "dewa_skip_new_rate": _rate(
                kernel_totals["normal_skip_new"], decisions
            ),
            "dewa_replace_old_rate": _rate(
                kernel_totals["normal_replace_old"], decisions
            ),
            "exception_product_rate": _rate(
                product_totals["exception_products"],
                product_totals["total_products"],
            ),
            "exception_routed_group_slot_rate": _rate(
                routed_slots, total_slots
            ),
            "exception_nonzero_partial_rate": _rate(
                nonzero_exception, total_slots
            ),
            "exception_skip_rate": _rate(
                kernel_totals["exception_skip_small"],
                nonzero_exception,
            ),
            "exception_fp_acc_request_rate": _rate(
                fp_adds, total_slots
            ),
            "final_dewa_flush_rate_per_output": _rate(
                final_flushes,
                product_totals["expected_output_slots"],
            ),
            "estimated_total_fp_acc_request_rate": _rate(
                total_fp_requests, total_slots
            ),
        },
    }
    return {"aggregate": aggregate, "layers": layers}


def validate_experiment_stats(stats):
    aggregate = stats["aggregate"]
    weight = aggregate["weight"]["counts"]
    activation = aggregate["activation"]
    product = aggregate["product_counts"]
    kernel = aggregate["kernel_counts"]
    errors = []

    if weight["total_blocks"] * FORMAT.block_size != weight["total_values"]:
        errors.append("weight block/value closure failed")
    if activation["counts"]["total_blocks"] * FORMAT.block_size != (
        activation["counts"]["total_values"]
    ):
        errors.append("activation block/value closure failed")
    if activation["max_encoded_outliers_per_block"] > (
        FORMAT.max_outliers_per_block
    ):
        errors.append("encoded Top-2 occupancy exceeded")

    if product["total_products"] != (
        product["normal_products"] + product["exception_products"]
    ):
        errors.append("normal/exception product closure failed")
    if product["expected_total_group_output_slots"] != (
        kernel["total_group_output_slots"]
    ):
        errors.append("total group-slot analytical/kernel mismatch")
    if product["expected_exception_routed_group_slots"] != (
        kernel["exception_routed_group_slots"]
    ):
        errors.append("routed group-slot analytical/kernel mismatch")
    if kernel["total_group_output_slots"] != (
        kernel["zero_normal_partial"]
        + kernel["normal_initial_load"]
        + kernel["normal_nonzero_decisions"]
    ):
        errors.append("normal group-slot partition failed")
    if kernel["normal_nonzero_decisions"] != (
        kernel["normal_skip_new"]
        + kernel["normal_replace_old"]
        + kernel["normal_add"]
    ):
        errors.append("DEWA decision closure failed")
    if kernel["exception_routed_group_slots"] != (
        kernel["zero_exception_partial"]
        + kernel["nonzero_exception_partial"]
    ):
        errors.append("exception routed-slot partition failed")
    if kernel["nonzero_exception_partial"] != (
        kernel["exception_send_no_reference"]
        + kernel["exception_skip_small"]
        + kernel["exception_send_referenced"]
    ):
        errors.append("exception admission partition failed")
    if kernel["exception_fp_acc_adds"] != (
        kernel["exception_send_no_reference"]
        + kernel["exception_send_referenced"]
    ):
        errors.append("exception FP-Acc request closure failed")
    if kernel["final_dewa_nonzero_flushes"] > (
        product["expected_output_slots"]
    ):
        errors.append("final DEWA flushes exceed output slots")

    return {"passed": not errors, "errors": errors}


## Synthetic reference and Triton parity tests


In [ ]:
_test_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Single-exponent weight quantization.
_weight = torch.tensor(
    [[0.0, -1.0, 0.5, 1.5] + [0.0] * 12],
    dtype=torch.float32,
    device=_test_device,
)
_weight_q = _quantize_weight_bfp_rows(_weight, FORMAT)
_weight_max = _weight.abs().amax()
_weight_exp = torch.floor(torch.log2(_weight_max))
_weight_step = torch.pow(
    torch.tensor(2.0, device=_test_device),
    _weight_exp - (FORMAT.mantissa_bits - 1),
)
_weight_reference = (
    torch.round(_weight / _weight_step)
    .clamp(
        -(1 << FORMAT.mantissa_bits) + 1,
        (1 << FORMAT.mantissa_bits) - 1,
    )
    * _weight_step
)
assert torch.equal(_weight_q, _weight_reference)

# Signed threshold and Top-2 occupancy for 0/1/2/3/16 candidates.
_signed = torch.zeros((1, 16), dtype=torch.float32, device=_test_device)
_signed[0, -1] = -100.0
(
    _signed_q,
    _signed_tags,
    _signed_t,
    _signed_counts,
    _signed_candidate_hist,
    _signed_encoded_hist,
    _signed_meta,
) = quantize_activation_top2(_signed, FORMAT, chunk_rows=1)
_signed_expected_t = (
    _signed.mean() + FORMAT.sigma_k * _signed.std(unbiased=False)
)
_abs_statistics_t = (
    _signed.abs().mean()
    + FORMAT.sigma_k * _signed.abs().std(unbiased=False)
)
assert torch.allclose(
    _signed_t, _signed_expected_t, rtol=1e-6, atol=1e-6
)
assert not torch.allclose(
    _signed_t, _abs_statistics_t, rtol=1e-6, atol=1e-6
)
assert _counts_to_dict(
    _signed_counts, ACTIVATION_COUNT_NAMES
)["encoded_outlier_values"] == 1
assert torch.equal(
    _unpack_bits_last_dim(_signed_tags, 16),
    _signed.abs() > _signed_t,
)

_crafted = torch.zeros((5, 16), dtype=torch.float32, device=_test_device)
_crafted[1, 0] = 1.0
_crafted[2, :2] = 1.0
_crafted[3, :3] = torch.tensor(
    [1.0, 2.0, 3.0], device=_test_device
)
_crafted[4, :] = torch.arange(
    1, 17, dtype=torch.float32, device=_test_device
)
(
    _crafted_q,
    _crafted_mask,
    _crafted_counts,
    _crafted_candidate_hist,
    _crafted_encoded_hist,
) = _quantize_activation_rows_with_threshold(
    _crafted, torch.tensor(0.5, device=_test_device), FORMAT
)
_crafted_dict = _counts_to_dict(
    _crafted_counts, ACTIVATION_COUNT_NAMES
)
_crafted_summary = _summarize_activation(
    _crafted_dict,
    _crafted_candidate_hist.detach().cpu().tolist(),
    _crafted_encoded_hist.detach().cpu().tolist(),
    include_tail=True,
)
assert _crafted_summary[
    "candidate_outliers_per_block_histogram"
][0] == 1
assert _crafted_summary[
    "candidate_outliers_per_block_histogram"
][1] == 1
assert _crafted_summary[
    "candidate_outliers_per_block_histogram"
][2] == 1
assert _crafted_summary[
    "candidate_outliers_per_block_histogram"
][3] == 1
assert _crafted_summary[
    "candidate_outliers_per_block_histogram"
][16] == 1
assert _crafted_summary[
    "encoded_outliers_per_block_histogram"
][0] == 1
assert _crafted_summary[
    "encoded_outliers_per_block_histogram"
][1] == 1
assert _crafted_summary[
    "encoded_outliers_per_block_histogram"
][2] == 3
assert _crafted_summary["max_encoded_outliers_per_block"] == 2
assert _crafted_dict["candidate_values"] == 22
assert _crafted_dict["encoded_outlier_values"] == 7
assert _crafted_dict["demoted_values"] == 15
assert _crafted_dict["cap_triggered_blocks"] == 2
assert _crafted_q[4, 13].item() == 14.0
assert _crafted_mask[3].detach().cpu().tolist() == (
    [False, True, True] + [False] * 13
)

# Equal magnitudes prefer lower K indices.
_tie_magnitude = torch.tensor(
    [[[5.0, 5.0, 5.0] + [0.0] * 13]],
    device=_test_device,
)
_tie_candidate = _tie_magnitude > 0
_tie_selected = _select_top2_candidates(
    _tie_magnitude, _tie_candidate, FORMAT
)
assert _tie_selected[0, 0, :3].detach().cpu().tolist() == [
    True,
    True,
    False,
]

# Packed tag order and non-byte-aligned round trip.
_tag_test = torch.tensor(
    [[0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1]],
    dtype=torch.bool,
    device=_test_device,
)
assert torch.equal(
    _unpack_bits_last_dim(_pack_bits_last_dim(_tag_test), 11),
    _tag_test,
)

# Threshold-before-chunking and chunk invariance.
torch.manual_seed(7)
_chunk_input = torch.randn(
    (5, 32), dtype=torch.float32, device=_test_device
)
_chunk_1 = quantize_activation_top2(
    _chunk_input, FORMAT, chunk_rows=1
)
_chunk_5 = quantize_activation_top2(
    _chunk_input, FORMAT, chunk_rows=5
)
assert torch.allclose(
    _chunk_1[2], _chunk_5[2], rtol=1e-6, atol=1e-6
)
for index in (0, 1, 3, 4, 5):
    assert torch.equal(_chunk_1[index], _chunk_5[index])

# DEWA boundaries and current numeric exponent.
_acc = torch.tensor(
    [[2.0**10, 2.0**10, 0.0]], device=_test_device
)
_partial = torch.tensor(
    [[2.0**2, 2.0**18, 2.0**-7]], device=_test_device
)
_updated, _normal_counts = _dewa_reference_update(
    _acc, _partial, 8, True
)
assert _updated[0, 0] == _acc[0, 0]
assert _updated[0, 1] == _partial[0, 1]
assert _updated[0, 2] == _partial[0, 2]
assert _normal_counts["normal_skip_new"].item() == 1
assert _normal_counts["normal_replace_old"].item() == 1

_cancel_acc = torch.tensor([[2.0**10]], device=_test_device)
_cancel_partial = torch.tensor(
    [[-(2.0**10) + 2.0**2]], device=_test_device
)
_cancel_acc, _ = _dewa_reference_update(
    _cancel_acc, _cancel_partial, 8, True
)
assert _cancel_acc.item() == 2.0**2

# Exception admission is one-sided and uses pre-update DEWA state.
_exception_old = torch.tensor(
    [[2.0**10, 2.0**10, 0.0, 2.0**10]],
    device=_test_device,
)
_exception_partial = torch.tensor(
    [[2.0**2, 2.0**3, 2.0**20, 0.0]],
    device=_test_device,
)
_exception_send, _exception_decision = (
    _exception_reference_decision(
        _exception_old, _exception_partial, 8, True
    )
)
assert _exception_send.detach().cpu().tolist() == [
    [False, True, True, False]
]
assert _exception_decision[
    "exception_skip_small"
].sum().item() == 1
assert _exception_decision[
    "exception_send_no_reference"
].sum().item() == 1

_pre_acc = torch.tensor([[2.0**2]], device=_test_device)
_same_scale_exception = torch.tensor(
    [[2.0**2]], device=_test_device
)
_pre_send, _ = _exception_reference_decision(
    _pre_acc, _same_scale_exception, 8, True
)
_post_acc, _ = _dewa_reference_update(
    _pre_acc,
    torch.tensor([[2.0**10]], device=_test_device),
    8,
    True,
)
_post_send, _ = _exception_reference_decision(
    _post_acc, _same_scale_exception, 8, True
)
assert _pre_send.item()
assert not _post_send.item()

_zero_send, _zero_decision = _exception_reference_decision(
    _pre_acc, torch.zeros_like(_pre_acc), 8, True
)
assert not _zero_send.item()
assert not _zero_decision["exception_nonzero"].item()

# Exact activation-only split identity with both approximations disabled.
_x_ref = torch.tensor(
    [[1.0, -2.0] * 16, [0.5, 1.0] * 16],
    dtype=torch.float16,
    device=_test_device,
)
_w_ref = torch.tensor(
    [
        [1.0, 0.5] * 16,
        [-1.0, 2.0] * 16,
        [0.25, -0.5] * 16,
    ],
    dtype=torch.float16,
    device=_test_device,
)
_x_tag_ref = torch.zeros_like(_x_ref, dtype=torch.bool)
_x_tag_ref[0, 1] = True
_x_tag_ref[0, 17] = True
_bias_ref = torch.tensor(
    [0.5, -1.0, 2.0],
    dtype=torch.float16,
    device=_test_device,
)
_exact_config = DEWAConfig(
    threshold_bits=8,
    exception_threshold_bits=8,
    enabled=False,
    exception_skip_enabled=False,
)
_split_y, _split_stats = reference_hybrid_linear(
    _x_ref,
    _x_tag_ref,
    _w_ref,
    _bias_ref,
    _exact_config,
)
_direct_y = F.linear(
    _x_ref.float(), _w_ref.float(), _bias_ref.float()
).half()
assert torch.equal(_split_y, _direct_y)
assert _split_stats[11].item() == 0

if torch.cuda.is_available():
    # CUDA parity uses valid Top-2 masks and exact integer-representable values.
    torch.manual_seed(11)
    _m, _n, _k = 5, 37, 64
    _x_cuda = (
        torch.randint(-4, 5, (_m, _k), device="cuda").to(torch.float16)
        / 4
    )
    _w_cuda = (
        torch.randint(-4, 5, (_n, _k), device="cuda").to(torch.float16)
        / 4
    )
    _bias_cuda = (
        torch.randint(-2, 3, (_n,), device="cuda").to(torch.float16)
        / 4
    )
    _normal_tags = torch.zeros(
        (_m, _k), dtype=torch.bool, device="cuda"
    )
    _top2_tags = _normal_tags.clone()
    for _row in range(_m):
        for _group_start in range(0, _k, 16):
            _top2_tags[_row, _group_start] = True
            if (_row + _group_start // 16) % 2:
                _top2_tags[_row, _group_start + 5] = True

    _configs = [
        _exact_config,
        *[
            DEWAConfig(
                threshold_bits=_threshold,
                exception_threshold_bits=_threshold,
                enabled=True,
                exception_skip_enabled=True,
            )
            for _threshold in T_SWEEP
        ],
    ]
    for _tags in (_normal_tags, _top2_tags):
        _packed = _pack_bits_last_dim(_tags)
        for _config in _configs:
            _cuda_stats = _empty_kernel_stats(torch.device("cuda"))
            _triton_y = fused_top2_dewa_linear(
                _x_cuda,
                _packed,
                _w_cuda,
                _bias_cuda,
                _config,
                _cuda_stats,
            )
            _reference_y, _reference_stats = reference_hybrid_linear(
                _x_cuda,
                _tags,
                _w_cuda,
                _bias_cuda,
                _config,
            )
            torch.cuda.synchronize()
            torch.testing.assert_close(
                _triton_y, _reference_y, rtol=0, atol=2**-8
            )
            assert torch.equal(_cuda_stats, _reference_stats), (
                asdict(_config),
                _named_stats(_cuda_stats, KERNEL_STAT_NAMES),
                _named_stats(_reference_stats, KERNEL_STAT_NAMES),
            )

    del (
        _x_cuda,
        _w_cuda,
        _bias_cuda,
        _normal_tags,
        _top2_tags,
        _packed,
        _cuda_stats,
        _triton_y,
        _reference_y,
        _reference_stats,
    )
    torch.cuda.empty_cache()

del (
    _weight,
    _weight_q,
    _weight_reference,
    _signed,
    _signed_q,
    _crafted,
    _crafted_q,
    _tie_magnitude,
    _tie_candidate,
    _tie_selected,
    _tag_test,
    _chunk_input,
    _acc,
    _partial,
    _updated,
    _x_ref,
    _w_ref,
    _x_tag_ref,
    _bias_ref,
    _split_y,
    _direct_y,
)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Synthetic Top-2 BiE + DEWA + conditional FP-ACC checks passed.")


## Dataset and non-overlapping PPL evaluation


In [ ]:
token = os.getenv('HF_TOKEN')
if not token:
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
    except Exception:
        token = None
if not token:
    token = getpass('HF_TOKEN: ')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token)
dataset = load_dataset(DATASET_ID, DATASET_CONFIG, split=SPLIT)
text = '\n\n'.join(dataset['text'])
input_ids = tokenizer(text, return_tensors='pt').input_ids
print(f'WikiText-2 {SPLIT} tokens: {input_ids.numel():,}')


@torch.inference_mode()
def evaluate_perplexity(
    model,
    input_ids,
    context_length,
    stride,
    drop_remainder,
    description,
):
    if stride != context_length:
        raise ValueError('Non-overlapping evaluation requires stride == context_length.')
    if not drop_remainder:
        raise ValueError('This evaluation requires drop_remainder=True.')
    if context_length > model.config.max_position_embeddings:
        raise ValueError('context_length exceeds the model context window.')

    device = next(model.parameters()).device
    sequence_length = input_ids.size(1)
    usable_length = sequence_length // context_length * context_length
    dropped_tokens = sequence_length - usable_length
    if usable_length == 0:
        raise ValueError('Input does not contain a complete context block.')

    total_nll = 0.0
    total_loss_tokens = 0
    total_blocks = usable_length // context_length

    torch.cuda.reset_peak_memory_stats(device)
    torch.cuda.synchronize(device)
    start_time = time.perf_counter()

    for begin in tqdm(
        range(0, usable_length, stride),
        total=total_blocks,
        desc=description,
    ):
        end = begin + context_length
        batch = input_ids[:, begin:end].to(device)
        labels = batch.clone()
        loss = model(batch, labels=labels, use_cache=False).loss
        loss_tokens = labels[:, 1:].numel()
        total_nll += loss.float().item() * loss_tokens
        total_loss_tokens += loss_tokens

    torch.cuda.synchronize(device)
    elapsed_seconds = time.perf_counter() - start_time
    mean_nll = total_nll / total_loss_tokens

    return {
        'mean_nll': mean_nll,
        'perplexity': float(torch.exp(torch.tensor(mean_nll))),
        'source_input_tokens': sequence_length,
        'used_input_tokens': usable_length,
        'dropped_input_tokens': dropped_tokens,
        'evaluated_blocks': total_blocks,
        'evaluated_tokens': total_loss_tokens,
        'elapsed_seconds': elapsed_seconds,
        'tokens_per_second': total_loss_tokens / elapsed_seconds,
        'peak_gpu_memory_gib': (
            torch.cuda.max_memory_allocated(device) / 2**30
        ),
    }


## Load once, quantize once, then sweep T


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("The full PPL evaluation requires an NVIDIA CUDA GPU.")

torch.manual_seed(0)
initial_dewa = DEWAConfig(
    threshold_bits=T_SWEEP[0],
    exception_threshold_bits=T_SWEEP[0],
    enabled=True,
    exception_skip_enabled=True,
)
initial_dewa.validate()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map=0,
    attn_implementation="eager",
    token=token,
)
model.eval()
model.config.use_cache = False

hybrid_layers = replace_linear_layers(
    model, FORMAT, initial_dewa
)
if not hybrid_layers:
    raise RuntimeError("No nn.Linear layers were replaced.")
if "lm_head" in hybrid_layers or not isinstance(model.lm_head, nn.Linear):
    raise RuntimeError("lm_head must remain an unwrapped FP16 nn.Linear.")
if len(hybrid_layers) != 224:
    raise RuntimeError(
        f"Expected 224 quantized decoder Linear layers, "
        f"got {len(hybrid_layers)}."
    )

parameter_dtypes = {
    parameter.dtype
    for parameter in model.parameters()
    if parameter.is_floating_point()
}
parameter_devices = {
    parameter.device.type for parameter in model.parameters()
}
assert parameter_dtypes == {torch.float16}, parameter_dtypes
assert parameter_devices == {"cuda"}, parameter_devices

torch.cuda.empty_cache()
print(f"Hybrid DEWA Linear layers: {len(hybrid_layers)}")
print("Weight: BFP4 G16 single E5")
print("Activation: Top-2-capped BiE4 G16 dual E5")
print("lm_head: FP16 (not quantized)")
print("Exception reference: pre-update DEWA accumulator")
print("Exception threshold: tied to normal T for this sweep")


## Run T=8-12, validate, save JSON, and build ZIP

Only the final conditional-skip design is evaluated. The existing capped Top-2 activation-BiE baseline PPL is recorded as the direct comparator; a separate no-skip FP-ACC PPL run is intentionally omitted.


In [ ]:
RESULT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
result_paths = []
sweep_summary = []

for threshold_bits in T_SWEEP:
    dewa_config = configure_dewa(
        hybrid_layers, threshold_bits=threshold_bits
    )
    metrics = evaluate_perplexity(
        model,
        input_ids,
        CONTEXT_LENGTH,
        STRIDE,
        DROP_REMAINDER,
        description=(
            f"Top-2 A-BiE4 DEWA+conditional-FPACC "
            f"T={threshold_bits}"
        ),
    )
    experiment_stats = export_experiment_stats(hybrid_layers)
    validation = validate_experiment_stats(experiment_stats)

    aggregate = experiment_stats["aggregate"]
    weight_counts = aggregate["weight"]["counts"]
    activation_counts = aggregate["activation"]["counts"]
    if metrics["evaluated_blocks"] != EXPECTED_EVALUATED_BLOCKS:
        validation["errors"].append(
            f"expected {EXPECTED_EVALUATED_BLOCKS} blocks, "
            f"got {metrics['evaluated_blocks']}"
        )
    if metrics["evaluated_tokens"] != EXPECTED_LOSS_TOKENS:
        validation["errors"].append(
            f"expected {EXPECTED_LOSS_TOKENS} loss tokens, "
            f"got {metrics['evaluated_tokens']}"
        )
    if weight_counts["total_values"] != EXPECTED_WEIGHT_VALUES:
        validation["errors"].append(
            "unexpected quantized weight value count"
        )
    if activation_counts["total_values"] != EXPECTED_ACTIVATION_VALUES:
        validation["errors"].append(
            "unexpected quantized activation value count"
        )
    if aggregate["activation"]["threshold_summary"]["count"] != (
        EXPECTED_LINEAR_LAYERS * EXPECTED_EVALUATED_BLOCKS
    ):
        validation["errors"].append(
            "unexpected activation threshold evaluation count"
        )
    if dewa_config.exception_threshold_bits != (
        dewa_config.threshold_bits
    ):
        validation["errors"].append("T_exc must equal T in this sweep")
    if not dewa_config.enabled:
        validation["errors"].append("DEWA must be enabled")
    if not dewa_config.exception_skip_enabled:
        validation["errors"].append(
            "conditional exception skip must be enabled"
        )
    validation["passed"] = not validation["errors"]
    if not validation["passed"]:
        raise RuntimeError(
            f"T={threshold_bits} validation failed: "
            f"{validation['errors']}"
        )

    result = {
        "model": MODEL_ID,
        "dataset": f"{DATASET_ID}/{DATASET_CONFIG}",
        "split": SPLIT,
        "method_label": (
            "W-BFP4 / Top-2-capped A-BiE4 G16 + DEWA + "
            "conditional exception FP-ACC"
        ),
        "quantization": (
            "single-exponent W-BFP4 and signed-mu3sigma "
            "Top-2-capped A-BiE4 fake quantization"
        ),
        "format": (
            f"Weight BFP4 (1S{FORMAT.mantissa_bits}M + one "
            f"E{FORMAT.shared_exponent_bits}, G{FORMAT.block_size}); "
            f"Activation BiE4 (1S{FORMAT.mantissa_bits}M + two "
            f"E{FORMAT.shared_exponent_bits} + capped type, "
            f"G{FORMAT.block_size})"
        ),
        "hybrid_config": asdict(FORMAT),
        "dewa_config": asdict(dewa_config),
        "kernel_config": asdict(KERNEL),
        "threshold_contract": {
            "domain": "activation only",
            "formula": "mean(X) + sigma_k * std(X)",
            "statistics_domain": "signed tensor values",
            "comparison": "candidate iff abs(X) > threshold",
            "std_correction": 0,
            "granularity": (
                "one complete nn.Linear input tensor per forward call"
            ),
            "computed_before_row_chunking": True,
        },
        "top2_contract": {
            "block_definition": (
                "contiguous Group-16 values along Linear K"
            ),
            "encoded_rule": (
                "up to two largest-magnitude threshold candidates"
            ),
            "tie_break": FORMAT.topk_tie_break,
            "demoted_rule": (
                "remaining candidates join normal before exponent selection"
            ),
            "maximum_encoded_outliers_per_block": (
                FORMAT.max_outliers_per_block
            ),
        },
        "routing_contract": {
            "weight_outlier_classification": False,
            "exception_if": "encoded activation type == outlier",
            "normal_partial": (
                "sum of normal activation products within one G16"
            ),
            "exception_partial": (
                "sum of zero-to-two encoded outlier products within one G16"
            ),
            "normal_destination": (
                "current-value DEWA normal accumulator"
            ),
            "dewa_exponent_source": (
                "current numeric accumulator leading exponent"
            ),
            "dewa_skip_rule": "E_normal_partial - E_acc(t) <= -T",
            "dewa_replace_rule": "E_normal_partial - E_acc(t) >= T",
            "exception_reference": (
                "pre-update DEWA normal accumulator E_acc(t)"
            ),
            "exception_skip_rule": (
                "nonzero reference and "
                "E_exception_partial - E_acc(t) <= -T_exc"
            ),
            "exception_no_reference_rule": (
                "send every nonzero exception partial to FP32 FP-ACC"
            ),
            "exception_large_side_rule": (
                "send to FP32 FP-ACC; never replace DEWA state"
            ),
            "exception_threshold_tied_to_normal_threshold": True,
            "recombination": (
                "FP16(DEWA_normal_acc + FP32_exception_acc + bias)"
            ),
            "k_processing_order": "strict increasing Group-16 order",
        },
        "storage_contract": {
            "packed_storage_implemented": False,
            "software_tag_transport": (
                "little-endian uint8 mask along K"
            ),
            "sparse_index_storage_implemented": False,
            "weight_effective_bits_per_value": WEIGHT_BITS_PER_VALUE,
            "activation_effective_bits_per_value": (
                ACTIVATION_BITS_PER_VALUE
            ),
            "tensor_threshold_metadata_excluded": True,
            "fake_quantized_values_stored_as": "FP16",
        },
        "quantized_scope": {
            "operations": "all decoder nn.Linear modules",
            "weights": "BFP4 single shared exponent",
            "activations": "Top-2-capped BiE4 dual shared exponent",
            "lm_head": FORMAT.quantize_lm_head,
            "attention_internal_matmul": False,
            "softmax": False,
            "layernorm": False,
        },
        "no_exception_skip_ppl_run": False,
        "no_exception_skip_note": (
            "Intentionally omitted by request; only the final "
            "conditional-skip path receives full PPL evaluation."
        ),
        "linear_output_dtype": "float16",
        "normal_accumulator_dtype": "float32 functional model",
        "exception_accumulator_dtype": "float32",
        "matmul_backend": "custom Triton tagged Group-16 kernel",
        "quantized_linear_layers": len(hybrid_layers),
        "attention_implementation": "eager",
        "context_length": CONTEXT_LENGTH,
        "stride": STRIDE,
        "evaluation_protocol": EVALUATION_PROTOCOL,
        "drop_remainder": DROP_REMAINDER,
        "fp16_baseline_perplexity": FP16_BASELINE_PPL,
        "top2_baseline_perplexity": TOP2_BASELINE_PPL,
        "top2_baseline_result": TOP2_BASELINE_RESULT,
        "delta_perplexity_vs_fp16": (
            metrics["perplexity"] - FP16_BASELINE_PPL
        ),
        "delta_perplexity_vs_top2": (
            metrics["perplexity"] - TOP2_BASELINE_PPL
        ),
        "experiment_stats": experiment_stats,
        "validation": validation,
        "gpu": torch.cuda.get_device_name(0),
        "cuda": torch.version.cuda,
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "triton": triton.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        **metrics,
    }

    output_path = RESULT_DIR / (
        "w-bfp4-a-bie4-top2cap-dewa-fpacc-excskip-"
        f"t{threshold_bits}-s2048.json"
    )
    output_path.write_text(
        json.dumps(result, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    result_paths.append(output_path)
    sweep_summary.append(
        {
            "threshold_bits": threshold_bits,
            "exception_threshold_bits": (
                dewa_config.exception_threshold_bits
            ),
            "perplexity": result["perplexity"],
            "delta_vs_top2": result["delta_perplexity_vs_top2"],
            "delta_vs_fp16": result["delta_perplexity_vs_fp16"],
            "dewa_oow_rate": aggregate["rates"]["dewa_oow_rate"],
            "exception_skip_rate": aggregate["rates"][
                "exception_skip_rate"
            ],
            "exception_fp_acc_request_rate": aggregate["rates"][
                "exception_fp_acc_request_rate"
            ],
            "estimated_total_fp_acc_request_rate": aggregate["rates"][
                "estimated_total_fp_acc_request_rate"
            ],
        }
    )
    print(
        json.dumps(
            sweep_summary[-1], indent=2, ensure_ascii=False
        )
    )
    print(f"Saved: {output_path.resolve()}")

with zipfile.ZipFile(
    ZIP_PATH, mode="w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for result_path in result_paths:
        archive.write(result_path, arcname=result_path.name)

print(json.dumps(sweep_summary, indent=2, ensure_ascii=False))
print(f"ZIP: {ZIP_PATH.resolve()}")


## Download all result JSON files as one ZIP


In [ ]:
if not ZIP_PATH.is_file():
    raise FileNotFoundError(f'Result ZIP does not exist: {ZIP_PATH}')

try:
    from google.colab import files
except ImportError:
    print(f'Not running in Colab. ZIP remains at: {ZIP_PATH.resolve()}')
else:
    files.download(str(ZIP_PATH))
